# Amazon Beauty RQ-VAE pipeline

End-to-end run on the Amazon Product Reviews (Beauty) dataset:
1. Download + embed item metadata (sentence-t5-base, 768d)
2. Train the RQ-VAE
3. Generate Semantic IDs

All artifacts land under `outputs/amazon_beauty_*` because `name: amazon_beauty` in the config drives the paths.

In [ ]:
import sys
from pathlib import Path

# This notebook lives in rq-vae/src/. cd up to rq-vae/ so config paths and
# `outputs/` resolve, and put rq-vae/ on sys.path so `import src.*` works.
ROOT = Path.cwd()
if ROOT.name == "src":
    ROOT = ROOT.parent
%cd $ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG = "amazon_beauty_config.yaml"

## 1. Build embeddings

In [ ]:
from src.amazon_beauty_dataloader import prepare
from src.config import load_config

cfg = load_config(CONFIG)
prepare(cfg, download=True)

## 2. Train the RQ-VAE

In [ ]:
from src.train import train

history = train(cfg)

In [ ]:
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs, [h["train_recon"] for h in history], label="train")
axes[0].plot(epochs, [h["recon_loss"] for h in history], label="eval")
axes[0].set_title("recon loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
for l in range(len(history[0]["utilization"])):
    axes[1].plot(epochs, [h["utilization"][l] for h in history], label=f"L{l}")
axes[1].set_title("codebook utilization"); axes[1].set_xlabel("epoch"); axes[1].legend()
axes[2].plot(epochs, [h["sid_unique_fraction"] for h in history])
axes[2].set_title("unique SID fraction"); axes[2].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 3. Generate Semantic IDs

In [ ]:
from src.generate_sids import generate

ckpt = Path(cfg["output"]["checkpoints_dir"]) / "best.pt"
generate(cfg, str(ckpt))

In [ ]:
import json
import pandas as pd

sids = pd.read_csv(cfg["output"]["sids_csv"])
print(f"{len(sids)} items")
display(sids.head(10))

with open(cfg["output"]["metrics_json"]) as f:
    print(json.dumps(json.load(f), indent=2))